# Demo 07d — Boosting

**Class 07 · E4** (slides: `slides/07_ensembles.md`)

Bagging trains its trees in parallel. **Boosting** trains them in sequence, and each new tree works on what the previous ones got wrong. AdaBoost re-weights the examples; gradient boosting fits the residuals.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor
from scipy.optimize import minimize_scalar
from sklearn.datasets import make_moons
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor

from faa import boundaries, data, demo, plotlib
from faa import galleries as ga
from faa import synthetic as sy
from faa.demo import show, slide
from faa.galleries import CLASS_SETS, MEMBERS, REG_SETS, board, family_figure, gain_figure, names_of
from faa.metrics import mae, mcc
from faa.plotlib import BLUE, GREEN, GREY, ORANGE, PURPLE, RED, SKY

demo.setup("07")
SEED = 0
N_JOBS = 8

X_tr, y_tr, X_te, y_te = sy.load("circles", SEED)
LIM = boundaries.plane_limits(X_tr, pad=0.25)

In [ ]:
slide("AdaBoost on Ten Points: Round by Round")
X10, y10 = make_moons(10, noise=0.2, random_state=2)  # a toy set chosen so that three stumps classify all ten points
t10 = 2 * y10 - 1  # AdaBoost works with labels -1 / +1 (class 1 of the deck is +1)
w = np.full(10, 0.1)
F10 = np.zeros(10)
rounds10 = []
for s in range(1, 4):
    stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X10, t10, sample_weight=w)
    h = stump.predict(X10)
    err = float(w[h != t10].sum())
    alpha = 0.5 * np.log((1 - err) / err)
    w_new = w * np.exp(-alpha * t10 * h)
    Z = w_new.sum()
    rounds10.append(
        (
            s,
            stump.tree_.feature[0],
            float(stump.tree_.threshold[0]),
            err,
            float(alpha),
            w.copy(),
            (w_new / Z).copy(),
            h.copy(),
        )
    )
    print(
        f"round {s}: stump on x{stump.tree_.feature[0] + 1} <= {stump.tree_.threshold[0]:.3f}; weighted error err = {err:.3f}; beta = 1/2 ln((1 - err)/err) = {alpha:.3f}; normaliser Z = {Z:.3f}"
    )
    w = w_new / Z
    F10 += alpha * h
show(pl.DataFrame({"i": range(1, 11), "x1": X10[:, 0], "x2": X10[:, 1], "y": t10}), 3)
for s, f, thr, err, alpha, w_before, w_after, h in rounds10:
    print(f"round {s}: weights before {np.round(w_before, 3).tolist()}")
    print(
        f"         wrong points {(np.flatnonzero(h != t10) + 1).tolist()}; weights after {np.round(w_after, 3).tolist()}"
    )
print(
    f"after round 1: the {int(np.sum(rounds10[0][7] != t10))} misclassified points hold {rounds10[0][6][rounds10[0][7] != t10].sum():.3f} of the weight (always 1/2: the next stump starts at the level of a coin)"
)
print(
    f"final vote sign(sum beta_b f_b) = {np.sign(F10).astype(int).tolist()}; true y = {t10.tolist()}; right on {int(np.sum(np.sign(F10) == t10))} of 10 points"
)

In [ ]:
slide("Where beta Comes From: the Exponential Loss")
s, f, thr, err, alpha, w_before, w_after, h = rounds10[0]
loss = lambda a: float(np.sum(w_before * np.exp(-a * t10 * h)))  # noqa: E731
res = minimize_scalar(loss, bounds=(0, 5), method="bounded", options={"xatol": 1e-10})
print(
    f"round 1: exponential loss of the weighted vote as a function of beta: L(beta) = sum_i w_i exp(-beta y_i f_b(x_i)); err = {err:.3f}"
)
print(
    f"numerical minimiser beta* = {res.x:.4f}; formula 1/2 ln((1 - err)/err) = {alpha:.4f}; L(beta*) = {res.fun:.4f}; 2 sqrt(err (1 - err)) = {2 * np.sqrt(err * (1 - err)):.4f}"
)
show(
    pl.DataFrame(
        {
            "alpha": [0.0, 0.25, 0.5, float(alpha), 1.0, 1.5],
            "L(beta)": [loss(a) for a in (0.0, 0.25, 0.5, alpha, 1.0, 1.5)],
        }
    ),
    4,
)

In [ ]:
# plot e4_beta
betas = np.linspace(0, 1.6, 200)
fg, ax = plotlib.fig("full", 1.1)
ax.plot(betas, [loss(a) for a in betas], color=BLUE)
ax.scatter([alpha], [loss(alpha)], color=RED, s=12, zorder=3)
ax.annotate(
    f"minimum at $\\beta$ = {alpha:.3f}",
    (alpha, loss(alpha)),
    xytext=(0.95, 0.95),
    fontsize=8,
    color=RED,
    arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.5},
)
ax.set(xlabel="member weight $\\beta$", ylabel="exponential loss")
plotlib.save(fg, "e4_beta")

In [ ]:
# plot e4_weights
fg, ax = plotlib.fig("full", 1.1)
pts = np.arange(1, 11)
ax.bar(pts - 0.2, w_before, width=0.38, color=GREY, label="round 1")
ax.bar(
    pts + 0.2,
    w_after,
    width=0.38,
    color=[RED if p != q else BLUE for p, q in zip(h, t10)],
    label="round 2 (red: was wrong)",
)
ax.set(xlabel="point $i$", ylabel="weight", xticks=pts)
ax.grid(axis="x", visible=False)
ax.legend(ncols=2, loc="upper right", fontsize=6.3, handlelength=1.2)
plotlib.save(fg, "e4_weights")

In [ ]:
slide("AdaBoost with Stumps: by Hand and in scikit-learn")


class HandAdaBoost:
    """Discrete AdaBoost (SAMME, 2 classes) with depth-1 trees. Labels are 0 / 1 outside, -1 / +1 inside.
    `weights_[s]` are the example weights AFTER round s (the ones that train round s + 1)."""

    def __init__(self, n_rounds: int = 50, max_depth: int = 1, seed: int = 0):
        self.n_rounds, self.max_depth, self.seed = n_rounds, max_depth, seed

    def fit(self, X, y):
        t = 2 * np.asarray(y) - 1
        w = np.full(len(t), 1 / len(t))
        self.stumps_, self.alphas_, self.errs_, self.weights_ = [], [], [], []
        for s in range(self.n_rounds):
            m = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.seed).fit(X, t, sample_weight=w)
            h = m.predict(X)
            err = float(np.clip(w[h != t].sum(), 1e-12, 1 - 1e-12))
            alpha = 0.5 * np.log((1 - err) / err)
            w = w * np.exp(-alpha * t * h)
            w /= w.sum()
            (
                self.stumps_.append(m),
                self.alphas_.append(float(alpha)),
                self.errs_.append(err),
                self.weights_.append(w.copy()),
            )
        return self

    def decision_function(self, X, upto: int | None = None):
        k = self.n_rounds if upto is None else upto
        return sum(a * m.predict(X) for a, m in zip(self.alphas_[:k], self.stumps_[:k]))

    def predict_proba(self, X, upto: int | None = None):
        p1 = 1 / (1 + np.exp(-2 * self.decision_function(X, upto)))  # the logistic link of the exponential loss
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)


class AtRound:
    """A view of a HandAdaBoost after `s` rounds, so that boundary_panel can draw it."""

    def __init__(self, model, s):
        self.model, self.s = model, s

    def predict_proba(self, X):
        return self.model.predict_proba(X, self.s)


ada = HandAdaBoost(50).fit(X_tr, y_tr)
sk = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=SEED).fit(
    X_tr, y_tr
)
print(
    f"by hand, 50 stumps: test MCC {mcc(y_te, ada.predict(X_te)):.3f}; AdaBoostClassifier {mcc(y_te, sk.predict(X_te)):.3f}; they agree on {np.mean(ada.predict(X_te) == sk.predict(X_te)):.3f} of the test points"
)
print(
    f"first beta by hand {np.round(ada.alphas_[:4], 3).tolist()}, scikit-learn {np.round(sk.estimator_weights_[:4], 3).tolist()} (no 1/2 factor there)"
)
print(f"weighted error: {', '.join(f'{e:.3f}' for e in ada.errs_[:5])}, ... {ada.errs_[-1]:.3f} at round 50")
rows = []
for s in (1, 2, 5, 50):
    wts = ada.weights_[s - 1]
    rows.append(
        (s, float(ada.alphas_[s - 1]), float(wts.max() * len(wts)), mcc(y_te, ada.decision_function(X_te, s) > 0))
    )
show(pl.DataFrame(rows, schema=["round b", "beta_b", "max weight / (1/n)", "test MCC"], orient="row"))

In [ ]:
# plot e4_adaboost
fg, axs = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.25))
fg.subplots_adjust(left=0.005, right=0.995, top=0.9, bottom=0.11, wspace=0.05)
for ax, s in zip(axs, (1, 2, 5, 50)):
    label = f"MCC {mcc(y_te, ada.decision_function(X_te, s) > 0):.2f}"
    boundaries.frontier_panel(ax, AtRound(ada, s), X_tr, y_tr, LIM, title=f"after round {s}", label=label, fontsize=6.3)
    ax.scatter(
        *X_tr.T,
        s=np.clip(120 * ada.weights_[s - 1], 0.05, 12),
        facecolor="none",
        edgecolor=RED,
        linewidth=0.25,
        zorder=4,
    )
plotlib.save(fg, "e4_adaboost")

In [ ]:
slide("Test MCC Against the Number of Rounds")


def ada_curve(seed, depth, T=300):
    a, c, b, d = sy.load("circles", seed)
    m = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=depth), n_estimators=T, random_state=seed).fit(
        a, c
    )
    return np.array([mcc(c, p) for p in m.staged_predict(a)]), np.array([mcc(d, p) for p in m.staged_predict(b)])


curves = {}
for depth in (1, 3):
    r = Parallel(n_jobs=N_JOBS)(delayed(ada_curve)(s, depth) for s in range(10))
    curves[depth] = (np.mean([x[0] for x in r], axis=0), np.mean([x[1] for x in r], axis=0))
show(
    pl.DataFrame(
        [
            (dp, T, float(tr[T - 1]), float(te[T - 1]))
            for dp, (tr, te) in curves.items()
            for T in (1, 5, 10, 50, 100, 300)
        ],
        schema=["tree depth", "rounds", "train MCC", "test MCC"],
        orient="row",
    )
)
for dp, (tr, te) in curves.items():
    print(
        f"depth {dp}: best test MCC {te.max():.3f} at {int(te.argmax()) + 1} rounds; at 300 rounds {te[-1]:.3f}, train {tr[-1]:.3f}"
    )

In [ ]:
# plot e4_rounds
fg, ax = plotlib.fig("full", 1.5)
T_axis = np.arange(1, 301)
for depth, col, name in ((1, BLUE, "stumps"), (3, ORANGE, "trees of depth 3")):
    tr, te = curves[depth]
    ax.plot(T_axis, te, color=col, label=f"{name}: test")
    ax.plot(T_axis, tr, color=col, ls="--", label=f"{name}: train")
ax.set(xscale="log", xlabel="boosting rounds $B$ (log scale)", ylabel="MCC (mean of 10 seeds)")
ax.legend(ncols=2, loc="lower right", fontsize=6.3, handlelength=1.6, columnspacing=0.8, labelspacing=0.25)
plotlib.save(fg, "e4_rounds")

In [ ]:
slide("The Margin of the Training Points")
t_tr = 2 * y_tr - 1
big = HandAdaBoost(300, max_depth=1).fit(X_tr, y_tr)
stump_out = np.array([m.predict(X_tr) for m in big.stumps_])  # 300 x n
cum = np.cumsum(np.array(big.alphas_)[:, None] * stump_out, axis=0)
norm = np.cumsum(big.alphas_)[:, None]
marg = {s: t_tr * cum[s - 1] / norm[s - 1] for s in (5, 50, 300)}
rows = [(s, float(np.mean(m <= 0)), float(np.min(m)), float(np.median(m))) for s, m in marg.items()]
show(
    pl.DataFrame(rows, schema=["rounds B", "share with margin <= 0", "smallest margin", "median margin"], orient="row")
)

In [ ]:
# plot e4_margin
fg, ax = plotlib.fig("col", 1.5)
for s, col in zip(marg, (SKY, BLUE, ORANGE)):
    m = np.sort(marg[s])
    ax.plot(m, np.arange(1, len(m) + 1) / len(m), color=col, label=f"$B$ = {s}")
ax.axvline(0, color="k", lw=0.6, ls=":")
ax.set(
    xlabel="normalised margin $y\\,F_B(x) / \\sum_b \\beta_b$", ylabel="share of training points below", xlim=(-1, 1)
)
ax.legend(loc="lower right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e4_margin")

In [ ]:
slide("Gradient Boosting for Squared Loss: Two Rounds by Hand")


class HandGB:
    """Gradient boosting for squared loss: F_0 = mean(y); round s fits a tree to the residuals y - F_{s-1} and adds eta times it."""

    def __init__(self, n_rounds: int = 100, eta: float = 0.1, max_depth: int = 3, seed: int = 0):
        self.n_rounds, self.eta, self.max_depth, self.seed = n_rounds, eta, max_depth, seed

    def fit(self, X, y):
        self.f0_ = float(np.mean(y))
        F = np.full(len(y), self.f0_)
        self.trees_, self.residuals_ = [], []
        for s in range(self.n_rounds):
            r = y - F  # the pseudo-residual of the squared loss: minus the gradient of (y - F)^2 / 2
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth, criterion="friedman_mse", random_state=self.seed
            ).fit(X, r)
            F = F + self.eta * tree.predict(X)
            self.trees_.append(tree), self.residuals_.append(r)
        return self

    def predict(self, X, upto: int | None = None):
        k = self.n_rounds if upto is None else upto
        return self.f0_ + self.eta * sum(t.predict(X) for t in self.trees_[:k]) if k else np.full(len(X), self.f0_)


rng = np.random.default_rng(SEED)
x8 = np.sort(rng.uniform(-3, 3, 8))
y8 = np.sin(1.5 * x8) + rng.normal(0, 0.3, 8)
ETA8 = 0.5
g8 = HandGB(2, eta=ETA8, max_depth=1).fit(x8[:, None], y8)
F0 = np.full(8, g8.f0_)
h1, h2 = g8.trees_[0].predict(x8[:, None]), g8.trees_[1].predict(x8[:, None])
F1 = F0 + ETA8 * h1
F2 = F1 + ETA8 * h2
show(
    pl.DataFrame(
        {
            "x": x8,
            "y": y8,
            "F0": F0,
            "r1 = y - F0": g8.residuals_[0],
            "h1": h1,
            "F1": F1,
            "r2 = y - F1": g8.residuals_[1],
            "h2": h2,
            "F2": F2,
        }
    ),
    3,
)
print(
    f"eta = {ETA8}, stumps; F0 = mean(y) = {g8.f0_:.3f}; split of round 1 at x <= {g8.trees_[0].tree_.threshold[0]:.3f}, of round 2 at x <= {g8.trees_[1].tree_.threshold[0]:.3f}"
)
print(
    f"training MSE: F0 {np.mean((y8 - F0) ** 2):.3f}, F1 {np.mean((y8 - F1) ** 2):.3f}, F2 {np.mean((y8 - F2) ** 2):.3f}"
)

In [ ]:
slide("Gradient Boosting on the Sawtooth: by Hand and in scikit-learn")
Xs_tr, ys_tr, Xs_te, ys_te = sy.load("sawtooth", SEED)
f_saw = sy.CURVES["sawtooth"]
hand = HandGB(200, eta=0.1, max_depth=3).fit(Xs_tr, ys_tr)
skg = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED).fit(Xs_tr, ys_tr)
print(
    f"test MAE by hand {mae(hand.predict(Xs_te), ys_te):.4f}, scikit-learn {mae(skg.predict(Xs_te), ys_te):.4f}; largest difference {np.max(np.abs(hand.predict(Xs_te) - skg.predict(Xs_te))):.1e}"
)
show(
    pl.DataFrame(
        [
            (s, float(np.mean((ys_tr - hand.predict(Xs_tr, s)) ** 2)), mae(hand.predict(Xs_te, s), ys_te))
            for s in (0, 1, 2, 5, 10, 30, 100, 200)
        ],
        schema=["rounds", "train MSE", "test MAE"],
        orient="row",
    ),
    4,
)

In [ ]:
# plot e4_residuals: F_s above, the residual and the tree fitted to it below
hand_r = HandGB(50, eta=0.3, max_depth=3).fit(Xs_tr, ys_tr)
xs = np.linspace(-3, 3, 300)[:, None]
fg, axs = plt.subplots(2, 4, figsize=(plotlib.FULL, 2.2), sharex=True)
order = np.argsort(Xs_tr[:, 0])
for j, s in enumerate((1, 2, 5, 50)):
    ax = axs[0, j]
    ax.scatter(Xs_tr[:, 0], ys_tr, s=3.5, color=ORANGE, edgecolor="white", linewidth=0.2, zorder=3)
    ax.plot(xs[:, 0], f_saw(xs[:, 0]), color=boundaries.LINE, lw=0.6, ls=":")
    ax.plot(xs[:, 0], hand_r.predict(xs, s), color=BLUE, lw=1.0)
    ax.set_xlabel("model $F$ and data", fontsize=6.3, labelpad=1.5)
    ax.set_title(f"after round {s}", fontsize=6.8, pad=2)
    ax = axs[1, j]
    r = hand_r.residuals_[s - 1]
    ax.scatter(Xs_tr[:, 0], r, s=3.5, color=GREY, edgecolor="white", linewidth=0.2, zorder=3)
    ax.plot(xs[:, 0], hand_r.trees_[s - 1].predict(xs), color=RED, lw=1.0)
    ax.axhline(0, color=boundaries.LINE, lw=0.4)
    ax.set_xlabel("residual and tree", fontsize=6.3, labelpad=1.5)
    for a in axs[:, j]:
        a.set(xticks=[], yticks=[])
        a.grid(False)
    axs[0, j].set_ylim(-1.6, 1.6)
    axs[1, j].set_ylim(-1.6, 1.6)
fg.subplots_adjust(wspace=0.06, hspace=0.38)
print(
    "residual sd per round (the part of y that the model has not explained):",
    ", ".join(f"s={s}: {hand_r.residuals_[s - 1].std():.3f}" for s in (1, 2, 5, 50)),
)
plotlib.save(fg, "e4_residuals")

In [ ]:
slide("Pseudo-Residuals of the Logistic Loss with jax.grad")


def logistic_loss(y, F):
    """-log-likelihood of a label y in {0, 1} for the log-odds F: softplus(F) - y F."""
    return jax.nn.softplus(F) - y * F


pseudo = jax.vmap(jax.grad(logistic_loss, argnums=1, allow_int=False), in_axes=(0, 0))
y5 = jnp.array([1.0, 1.0, 0.0, 0.0, 1.0])
F5 = jnp.array([-2.0, 0.0, 0.0, 1.5, 3.0])  # current log-odds
auto = -np.asarray(pseudo(y5, F5))  # r = - dL/dF
closed = np.asarray(y5 - jax.nn.sigmoid(F5))  # y - p
show(
    pl.DataFrame(
        {
            "y": np.asarray(y5),
            "F": np.asarray(F5),
            "p = sigmoid(F)": np.asarray(jax.nn.sigmoid(F5)),
            "r (jax.grad)": auto,
            "y - p": closed,
        }
    ),
    4,
)
Fbig = jnp.linspace(-6, 6, 1001)
auto_big = -np.asarray(jax.vmap(jax.grad(logistic_loss, argnums=1))(jnp.ones(1001), Fbig))
print(
    f"largest difference between jax.grad and y - p over F in [-6, 6]: {np.max(np.abs(auto_big - np.asarray(1 - jax.nn.sigmoid(Fbig)))):.1e}"
)
print(
    "squared loss: r = y - F (the residual); absolute loss: r = sign(y - F); logistic loss: r = y - p; the tree of round s is fitted to r in every case"
)
print(
    f"abs loss for the same F as a regression target y = 1, F = 0.2: r = {np.sign(1.0 - 0.2):+.0f}; squared loss: r = {1.0 - 0.2:+.1f}"
)

In [ ]:
slide("Shrinkage and the Number of Trees")


def lr_curves(seed, etas=(1.0, 0.3, 0.1, 0.03, 0.01), T=500):
    a, b, c, d = sy.load("sawtooth", seed)
    return {
        eta: np.array(
            [
                mae(p, d)
                for p in GradientBoostingRegressor(n_estimators=T, learning_rate=eta, max_depth=3, random_state=seed)
                .fit(a, b)
                .staged_predict(c)
            ]
        )
        for eta in etas
    }


lc = Parallel(n_jobs=N_JOBS)(delayed(lr_curves)(s) for s in range(5))
etas = list(lc[0])
mean_curve = {eta: np.mean([r[eta] for r in lc], axis=0) for eta in etas}
show(
    pl.DataFrame(
        [
            (eta, float(c[9]), float(c[99]), float(c[499]), float(c.min()), int(c.argmin()) + 1)
            for eta, c in mean_curve.items()
        ],
        schema=["eta", "10 trees", "100 trees", "500 trees", "best MAE", "at tree"],
        orient="row",
    )
)

In [ ]:
# plot e4_lr_grid
fg, ax = plotlib.fig("full", 1.5)
for eta, col in zip(etas, (RED, ORANGE, GREEN, BLUE, PURPLE)):
    ax.plot(np.arange(1, 501), mean_curve[eta], color=col, label=f"$\\eta$ = {eta:g}")
ax.set(xscale="log", xlabel="trees $B$ (log scale)", ylabel="test MAE")
ax.legend(ncols=5, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.3, handlelength=1.2, columnspacing=0.8)
plotlib.save(fg, "e4_lr_grid")

In [ ]:
slide("Early Stopping on a Validation Set")
Xh, yh = data.load_heart()


def early_run(seed, T=400):
    a, b, c, d = train_test_split(Xh, yh, test_size=0.4, stratify=yh, random_state=seed)
    a, v, c, e = train_test_split(a, c, test_size=0.25, stratify=c, random_state=seed)  # 60 / 20 / 20
    m = GradientBoostingClassifier(n_estimators=T, learning_rate=0.1, max_depth=3, random_state=seed).fit(a, c)
    curve = lambda X_, y_: np.array([log_loss(y_, p, labels=[0, 1]) for p in m.staged_predict_proba(X_)])  # noqa: E731
    es = GradientBoostingClassifier(
        n_estimators=T, learning_rate=0.1, max_depth=3, n_iter_no_change=10, validation_fraction=0.25, random_state=seed
    ).fit(a, c)
    return curve(a, c), curve(v, e), curve(b, d), es.n_estimators_, mcc(d, es.predict(b)), mcc(d, m.predict(b))


er = Parallel(n_jobs=N_JOBS)(delayed(early_run)(s) for s in range(10))
tr_c, va_c, te_c = (np.mean([r[i] for r in er], axis=0) for i in range(3))
best_s = int(va_c.argmin()) + 1
print("heart disease (303 patients), gradient boosting eta = 0.1, depth 3, 60/20/20 split, mean of 10 splits")
show(
    pl.DataFrame(
        [(s, float(tr_c[s - 1]), float(va_c[s - 1]), float(te_c[s - 1])) for s in (10, 25, best_s, 100, 200, 400)],
        schema=["trees B", "train log loss", "validation log loss", "test log loss"],
        orient="row",
    )
)
print(
    f"the mean validation loss is lowest at B = {best_s}; the test loss at that B is {te_c[best_s - 1]:.3f}, at B = 400 it is {te_c[-1]:.3f}"
)
print(
    f"n_iter_no_change = 10 (scikit-learn, its own validation share of 25 %) stops at {np.mean([r[3] for r in er]):.1f} trees on average (min {min(r[3] for r in er)}, max {max(r[3] for r in er)}); test MCC {np.mean([r[4] for r in er]):.3f} against {np.mean([r[5] for r in er]):.3f} for all 400 trees"
)

In [ ]:
# plot e4_early
fg, ax = plotlib.fig("full", 1.5)
ax.plot(np.arange(1, 401), tr_c, color=BLUE, label="train")
ax.plot(np.arange(1, 401), va_c, color=ORANGE, label="validation")
ax.plot(np.arange(1, 401), te_c, color=GREY, ls="--", label="test")
ax.axvline(best_s, color=RED, lw=0.8, ls=":")
ax.annotate(
    f"stop at $B$ = {best_s}",
    (best_s, va_c[best_s - 1]),
    xytext=(1.6, 0.85),
    fontsize=6.5,
    color=RED,
    arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.5},
)
ax.set(xscale="log", xlabel="trees $B$ (log scale)", ylabel="log loss", ylim=(0, 1.0))
ax.legend(ncols=3, loc="upper right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e4_early")

In [ ]:
slide("Modern Libraries: One Real Set, Four Implementations")
Xc, yc = data.load_california()
Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)
THREADS = 4
IMPLS = {
    "GradientBoosting (scikit-learn)": lambda: GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED
    ),
    "HistGradientBoosting (scikit-learn)": lambda: HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=3, max_leaf_nodes=None, random_state=SEED
    ),
    "XGBoost": lambda: XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, tree_method="hist", n_jobs=THREADS, random_state=SEED
    ),
    "LightGBM": lambda: LGBMRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, num_leaves=8, n_jobs=THREADS, random_state=SEED, verbose=-1
    ),
}
rows = []
for name, make in IMPLS.items():
    fits, preds = [], []
    for rep in range(1 if name.startswith("GradientBoosting") else 3):
        with threadpool_limits(THREADS):
            m = make()
            t0 = time.perf_counter()
            m.fit(Xa, ya)
            fits.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            p = m.predict(Xb)
            preds.append(time.perf_counter() - t0)
    rows.append((name, float(np.median(fits)), float(np.median(preds)), mae(p, yb)))
timing = pl.DataFrame(rows, schema=["implementation", "fit s", "predict s", "test MAE"], orient="row")
show(timing)
ref = timing["fit s"][0]
print(
    f"california housing: {len(ya)} training rows, {Xa.shape[1]} features, 200 trees of depth 3, eta = 0.1, {THREADS} threads (median of 3 runs, one run for the slow GradientBoosting; times depend on the machine)"
)
print(
    "fit time relative to GradientBoosting: "
    + ", ".join(
        f"{n.split(' ')[0]} {ref / t:.0f}x faster" for n, t in zip(timing["implementation"], timing["fit s"]) if t < ref
    )
)
print("CatBoost is not installed: it is on the slides only (ordered target statistics for categorical features)")

In [ ]:
# plot e4_libs
fg, ax = plotlib.fig("full", 1.1)
names = [n.split(" ")[0] for n in timing["implementation"]]
ax.barh(names[::-1], timing["fit s"].to_list()[::-1], color=ORANGE, height=0.6)
for y, (t, e) in enumerate(zip(timing["fit s"].to_list()[::-1], timing["test MAE"].to_list()[::-1])):
    ax.text(t * 1.1, y, f"{t:.2f} s, MAE {e:.3f}", fontsize=6.2, va="center")
ax.set(xscale="log", xlabel="fit time, seconds (log scale; typical)", xlim=(0.03, 1500))
ax.grid(axis="y", visible=False)
fg.subplots_adjust(left=0.25, right=0.98, bottom=0.3)
plotlib.save(fg, "e4_libs")

## Failure case: label noise

A share of the training labels is flipped and the test labels stay clean.

In [ ]:
slide("Failure Case: Boosting and Label Noise (no slide)")


def noise_run(flip, seed):
    X_, y_ = make_moons(3400, noise=0.2, random_state=seed)
    a, b, c, d = X_[:400], X_[400:], y_[:400], y_[400:]
    rng = np.random.default_rng(seed + 100)
    c = np.where(rng.random(len(c)) < flip, 1 - c, c)  # flip a share of the training labels, test labels stay clean
    ms = {
        "AdaBoost, stumps": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=seed
        ),
        "AdaBoost, depth 3": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3), n_estimators=200, random_state=seed
        ),
        "gradient boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=seed),
        "random forest": RandomForestClassifier(n_estimators=200, random_state=seed),
        "one tree": DecisionTreeClassifier(random_state=seed),
    }
    with threadpool_limits(1):
        return [(flip, n, mcc(d, m.fit(a, c).predict(b))) for n, m in ms.items()]


FLIPS = (0.0, 0.05, 0.1, 0.2, 0.3)
nr = Parallel(n_jobs=N_JOBS)(delayed(noise_run)(f, s) for f in FLIPS for s in range(5))
nl = pl.DataFrame([r for run in nr for r in run], schema=["flip", "model", "MCC"], orient="row")
nw = nl.group_by("flip", "model").agg(pl.col("MCC").mean()).pivot(on="model", index="flip", values="MCC").sort("flip")
print("share of training labels flipped; clean test labels; mean test MCC of 5 seeds:")
show(nw)

In [ ]:
# plot e4_noise
fg, ax = plotlib.fig("full", 1.25)
for name, col in (
    ("random forest", GREEN),
    ("gradient boosting", ORANGE),
    ("AdaBoost, depth 3", RED),
    ("AdaBoost, stumps", BLUE),
):
    ax.plot(nw["flip"], nw[name], color=col, marker="o", ms=2.5, label=name)
ax.set(xlabel="share of flipped training labels", ylabel="test MCC")
ax.legend(ncols=2, loc="lower left", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e4_noise")

## Boosting against everything so far

The same figures with AdaBoost, gradient boosting and XGBoost added.

In [ ]:
slide("Boosting: the Board")
F = ["shallow", "voting", "bagging", "boosting"]
clf = ga.evaluate(CLASS_SETS, names_of("c", F))
show(ga.wide(clf, names_of("c", F), CLASS_SETS))

In [ ]:
reg = ga.evaluate(REG_SETS, names_of("r", F))
show(ga.wide(reg, names_of("r", F), REG_SETS))

In [ ]:
trees_c, trees_r = ga.BAGGING_C + ga.BOOSTING_C, ga.BAGGING_R + ga.BOOSTING_R
gain_c = ga.gain(clf, CLASS_SETS, trees_c, ga.SHALLOW_C + ga.VOTING_C, "c")
gain_r = ga.gain(reg, REG_SETS, trees_r, ga.SHALLOW_R + ga.VOTING_R, "r")
show(gain_c)
show(gain_r)

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C, "trees": ga.BAGGING_C + ga.BOOSTING_C}
plotlib.save(family_figure("spiral", rows, {"voting": MEMBERS["c"]}), "e4_gallery_spiral")

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C, "trees": ga.BAGGING_C + ga.BOOSTING_C}
plotlib.save(family_figure("checkerboard", rows, {"voting": MEMBERS["c"]}), "e4_gallery_checker")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R, "trees": ga.BAGGING_R + ga.BOOSTING_R}
plotlib.save(family_figure("sawtooth", rows, {"voting": MEMBERS["r"]}), "e4_gallery_saw")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R, "trees": ga.BAGGING_R + ga.BOOSTING_R}
plotlib.save(family_figure("chirp", rows, {"voting": MEMBERS["r"]}), "e4_gallery_chirp")

In [ ]:
plotlib.save(board(clf, "c", CLASS_SETS), "e4_board_c")

In [ ]:
plotlib.save(board(reg, "r", REG_SETS), "e4_board_r")

In [ ]:
plotlib.save(gain_figure(gain_c, "gain in MCC: best tree ensemble over shallow and voting"), "e4_gain_c")

In [ ]:
plotlib.save(gain_figure(gain_r, "gain in MAE: best tree ensemble over shallow and voting"), "e4_gain_r")